# Mean–CVaR efficient frontier

**CPU-only counterpart of an NVIDIA cuFOLIO notebook.** The matching unmodified GPU notebook is in `../upstream_notebooks/`. This version uses deterministic synthetic one-minute bars so it runs on GitHub Actions without NVIDIA infrastructure.

The frontier solves several CPU Mean–CVaR problems at increasing risk-aversion levels. It is intentionally small for normal GitHub-hosted runners.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate / "src"))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cufolio_cpu.returns import daily_returns_from_minute_bars
from cufolio_cpu.synthetic import synthetic_minute_bars

In [ ]:
bars = synthetic_minute_bars(sessions=45, seed=42)
daily_log, daily_simple = daily_returns_from_minute_bars(bars)
print(f"{len(bars):,} minute bars -> {daily_simple.shape[0]} daily sessions x {daily_simple.shape[1]} assets")
daily_simple.tail()

In [ ]:
from cufolio_cpu.optimize import efficient_frontier

frontier = efficient_frontier(daily_simple, [0.5, 1.0, 2.0, 5.0, 10.0], max_weight=0.30)
frontier[["risk_aversion", "expected_return", "cvar", "status"]]

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(frontier["cvar"], frontier["expected_return"], marker="o")
for row in frontier.itertuples():
    plt.annotate(str(row.risk_aversion), (row.cvar, row.expected_return))
plt.xlabel("Historical CVaR loss")
plt.ylabel("Expected daily return")
plt.title("CPU Mean–CVaR frontier")
plt.show()